In [1]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import DebertaV2Tokenizer, DebertaV2ForSequenceClassification
from transformers import DataCollatorWithPadding, get_scheduler
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm

# 1️⃣ Load dataset
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

# 2️⃣ Basic text cleanup (optional)
train_df['text'] = train_df['text'].astype(str)
test_df['text'] = test_df['text'].astype(str)

# 3️⃣ Tokenizer
tokenizer = DebertaV2Tokenizer.from_pretrained('microsoft/deberta-v3-small')

# 4️⃣ Custom Dataset class
class TweetDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encodings = self.tokenizer(
            text,
            truncation=True,
            padding=False,
            max_length=128
        )
        if self.labels is not None:
            encodings['labels'] = int(self.labels[idx])
        return {k: torch.tensor(v) for k, v in encodings.items()}

# 5️⃣ Create Datasets and Loaders
train_dataset = TweetDataset(train_df['text'].values, train_df['target'].values, tokenizer)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=data_collator)

# 6️⃣ Model setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DebertaV2ForSequenceClassification.from_pretrained('microsoft/deberta-v3-small', num_labels=2)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
num_epochs = 2
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler(
    name='linear',
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

# 7️⃣ Training loop
model.train()
progress_bar = tqdm(range(num_training_steps))
for epoch in range(num_epochs):
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

# 8️⃣ Evaluation (on training data for demo)
model.eval()
preds, labels = [], []
for batch in train_loader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
    preds.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())
    labels.extend(batch['labels'].cpu().numpy())

acc = accuracy_score(labels, preds)
print(f'Training Accuracy: {acc:.4f}')

# 9️⃣ Generate predictions for test set
test_dataset = TweetDataset(test_df['text'].values, labels=None, tokenizer=tokenizer)
test_loader = DataLoader(test_dataset, batch_size=16, collate_fn=data_collator)

model.eval()
test_preds = []
for batch in test_loader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
    test_preds.extend(torch.argmax(outputs.logits, dim=-1).cpu().numpy())

# 🔟 Create submission file
submission = pd.DataFrame({
    'id': test_df['id'],
    'target': test_preds
})
submission.to_csv('submission.csv', index=False)
print('✅ submission.csv saved successfully!')

c:\Users\ssing\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\ssing\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ssing\.cache\huggingface\hub\models--microsoft--deberta-v3-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as 

Training Accuracy: 0.8856
✅ submission.csv saved successfully!
